# Gaussian GRAPE Stage 3: Endpoint Clustering

Stage 2 measured restart-level success, trap ratios, fidelity trends, and optimizer effort.
Stage 3 estimates distinct optimized endpoint families in the 8-dimensional endpoint parameter landscape.

This notebook does not run new optimization and does not modify the physics model. It clusters already optimized
endpoints:

```python
x = [A0, width0_ns, A1, width1_ns, A2, width2_ns, A3, width3_ns]
```

A cluster is an empirical endpoint family found by the optimizer, not a rigorous proof of a mathematical optimum.
High-fidelity clusters suggest successful solution branches; low-fidelity clusters suggest trapped or failed endpoint
families. Sign-related clusters may be physically equivalent and should be interpreted carefully.

In [ ]:
import json
import math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from sklearn.cluster import DBSCAN, AgglomerativeClustering
    from sklearn.decomposition import PCA
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import StandardScaler
    SKLEARN_AVAILABLE = True
    SKLEARN_IMPORT_ERROR = None
except Exception as exc:
    SKLEARN_AVAILABLE = False
    SKLEARN_IMPORT_ERROR = exc


STAGE2_ANALYSIS_DIR = Path("PiecewiseGRAPEAnalysis_stage2_selected_v1")
if not STAGE2_ANALYSIS_DIR.exists() and Path("Scripts/PiecewiseGRAPEAnalysis_stage2_selected_v1").exists():
    STAGE2_ANALYSIS_DIR = Path("Scripts/PiecewiseGRAPEAnalysis_stage2_selected_v1")

STAGE3_OUT = Path("PiecewiseGRAPEAnalysis_stage3_clusters_selected_v1")
if STAGE2_ANALYSIS_DIR.parts and STAGE2_ANALYSIS_DIR.parts[0] == "Scripts":
    STAGE3_OUT = Path("Scripts/PiecewiseGRAPEAnalysis_stage3_clusters_selected_v1")

RESTART_TABLE_PATH = STAGE2_ANALYSIS_DIR / "tables" / "restart_records.csv"

SUCCESS_THRESHOLD = 0.999
N_ETA_EXPECTED = None

AMP_MIN = -10.0
AMP_MAX = 10.0
WIDTH_MIN_NS = 0.05 * 10.0
WIDTH_MAX_NS = 1.50 * 10.0
CENTER_MIN_NS = 0.0
CENTER_MAX_NS = 10.0

# Filled from Stage 2 restart_records.csv. Supports Gaussian endpoints and piecewise 4-channel time-bin endpoints.
FEATURE_COLUMNS = None
AMP_COLUMNS = []
WIDTH_COLUMNS = []
CENTER_COLUMNS = []

# Match the project notebook's 0.1-GHz unit convention for eta/(2*pi*GHz) labels.
GHz = globals().get("GHz", 1e8)
ns = globals().get("ns", 1e-9)

figure_width = 10
figure_height = 6
FIGURE_STATUS = []


## Helpers and Path Resolution

The notebook can be opened from the project root or from `Scripts/`. It prefers the cleaned Stage 2 restart table
and only falls back to raw Stage 1 JSON files if that table is unavailable.

In [ ]:
def require_sklearn():
    if not SKLEARN_AVAILABLE:
        print("Stage 3 clustering requires scikit-learn.")
        print(f"Import error: {SKLEARN_IMPORT_ERROR}")
        raise ImportError("Stage 3 clustering requires scikit-learn.")


def resolve_existing_path(path):
    path = Path(path)
    if path.exists():
        return path
    scripts_path = Path("Scripts") / path
    if scripts_path.exists():
        return scripts_path
    return path


def resolve_stage2_dir(stage2_dir):
    return resolve_existing_path(stage2_dir)


def resolve_stage3_out(out_dir, stage2_dir=None):
    out_dir = Path(out_dir)
    if out_dir.is_absolute():
        return out_dir
    if out_dir.parts and out_dir.parts[0] == "Scripts":
        return out_dir
    if stage2_dir is not None:
        stage2_dir = Path(stage2_dir)
        if stage2_dir.parts and stage2_dir.parts[0] == "Scripts":
            return Path("Scripts") / out_dir
    if not Path("GaussianGRAPEAnalysis_stage2_rich_gaussian_sanity_v1").exists() and Path("Scripts/GaussianGRAPEAnalysis_stage2_rich_gaussian_sanity_v1").exists():
        return Path("Scripts") / out_dir
    return out_dir


def safe_float(value, default=np.nan):
    try:
        if value is None:
            return default
        value = float(value)
        return value if math.isfinite(value) else default
    except Exception:
        return default


def safe_int(value, default=0):
    try:
        if value is None:
            return default
        if isinstance(value, float) and not math.isfinite(value):
            return default
        return int(value)
    except Exception:
        return default


def safe_bool(value, default=False):
    if isinstance(value, bool):
        return value
    if value is None:
        return default
    if isinstance(value, str):
        value = value.strip().lower()
        if value in {"true", "1", "yes", "y"}:
            return True
        if value in {"false", "0", "no", "n"}:
            return False
        return default
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass
    return bool(value)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(key): make_json_safe(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(value) for value in obj]
    if isinstance(obj, np.ndarray):
        return make_json_safe(obj.tolist())
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        value = float(obj)
        return value if math.isfinite(value) else None
    if isinstance(obj, (np.complexfloating, complex)):
        return {"real": make_json_safe(np.real(obj)), "imag": make_json_safe(np.imag(obj))}
    if isinstance(obj, Path):
        return str(obj)
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


def ensure_output_dirs(out_dir):
    out_dir = Path(out_dir)
    tables_dir = out_dir / "tables"
    figures_dir = out_dir / "figures"
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    return tables_dir, figures_dir


def save_figure(fig, path, note=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight", dpi=160)
    FIGURE_STATUS.append({"figure": path.name, "path": str(path), "created": True, "note": note})
    plt.show()
    return path


def record_skipped_figure(name, reason):
    FIGURE_STATUS.append({"figure": name, "path": None, "created": False, "note": reason})
    print(f"Skipping {name}: {reason}")


def eta_figure_suffix(clustered_df, eta_index):
    eta_values = sorted(pd.to_numeric(clustered_df["eta_index"], errors="coerce").dropna().astype(int).unique())
    eta_index = int(eta_index)
    if eta_values and eta_index == eta_values[0]:
        return f"eta_{eta_index:03d}"
    if eta_values and eta_index == eta_values[-1]:
        return f"eta_{eta_index:03d}"
    mid_eta = eta_values[len(eta_values) // 2] if eta_values else eta_index
    if eta_index == mid_eta:
        return "mid_eta"
    return f"eta_{eta_index:03d}"

## Load Stage 2 Restart Table

The main input is `GaussianGRAPEAnalysis_stage2_final_v1/tables/restart_records.csv`. Rows with missing optimized
endpoint parameters are excluded from clustering and counted in the validation report.

In [ ]:
def parameter_kind(column):
    if column.startswith("A") or column.endswith("_A"):
        return "amp"
    if "width" in column:
        return "width"
    if "center" in column:
        return "center"
    if column.startswith("param_"):
        return "generic"
    return "other"


def infer_feature_columns_from_dataframe(df):
    metadata_columns = {
        "eta_index", "eta", "eta_over_2pi_GHz", "restart", "seed", "seed_type", "seed_subindex",
        "used_physical_seed", "initial_fidelity", "final_fidelity", "valid_final_fidelity",
        "success_recorded", "success_analysis", "trap_analysis", "stop_reason", "stop_reason_sanitized",
        "count", "ended_at_bounds", "invalid_gradient_evals", "trace_len", "trace_initial",
        "trace_final", "trace_best", "trace_improvement", "trace_monotone_nondec", "pulse_family",
        "optimizer", "objective", "target", "T", "T_ns", "L", "max_iter", "eps0", "dx0",
        "improvement_threshold", "success_threshold", "optimizer_success_threshold", "n_optimized_parameters",
        "n_channels", "n_gaussians_per_channel", "params_per_gaussian", "n_time_bins", "eta_dir", "source_file",
    }
    candidates = []
    for column in df.columns:
        if column in metadata_columns:
            continue
        if column.startswith("best_") or column.startswith("centroid_") or column.startswith("rep_"):
            continue
        if pd.api.types.is_numeric_dtype(df[column]) or pd.to_numeric(df[column], errors="coerce").notna().any():
            kind = parameter_kind(column)
            if kind in {"amp", "width", "center", "generic"}:
                candidates.append(column)
    return candidates


def set_feature_columns(feature_columns):
    global FEATURE_COLUMNS, AMP_COLUMNS, WIDTH_COLUMNS, CENTER_COLUMNS
    FEATURE_COLUMNS = list(feature_columns)
    AMP_COLUMNS = [col for col in FEATURE_COLUMNS if parameter_kind(col) == "amp"]
    WIDTH_COLUMNS = [col for col in FEATURE_COLUMNS if parameter_kind(col) == "width"]
    CENTER_COLUMNS = [col for col in FEATURE_COLUMNS if parameter_kind(col) == "center"]
    return FEATURE_COLUMNS


def extract_x_opt_scaled_from_record(record):
    x_opt_scaled = record.get("x_opt_scaled")
    if isinstance(x_opt_scaled, (list, tuple)) and len(x_opt_scaled) > 0:
        return [safe_float(value) for value in x_opt_scaled]

    c_opt = record.get("c_opt")
    try:
        c_array = np.asarray(c_opt, dtype=float)
        if c_array.shape == (4, 2):
            return [
                safe_float(value) / (ns if column == 1 else 1.0)
                for row in c_array
                for column, value in enumerate(row)
            ]
        if c_array.ndim == 3 and c_array.shape[0] == 4 and c_array.shape[2] >= 3:
            return [
                safe_float(value) / (ns if param_index in {1, 2} else 1.0)
                for channel in range(c_array.shape[0])
                for pulse_index in range(c_array.shape[1])
                for param_index, value in enumerate(c_array[channel, pulse_index, :3])
            ]
        if c_array.ndim == 2 and c_array.shape[0] == 4 and c_array.shape[1] > 2:
            return [safe_float(value) for value in c_array.reshape(-1)]
    except Exception:
        pass

    n_params = safe_int(record.get("n_optimized_parameters"), default=0)
    return [np.nan] * n_params if n_params > 0 else []


def parameter_names_for_record(record, x_values):
    n_params = len(x_values)
    pulse_family = str(record.get("pulse_family", ""))
    n_channels = safe_int(record.get("n_channels"), default=4)
    n_gaussians = safe_int(record.get("n_gaussians_per_channel"), default=0)
    params_per_gaussian = safe_int(record.get("params_per_gaussian"), default=0)
    if n_params == 8 and (not n_gaussians or params_per_gaussian in {0, 2}):
        return ["A0", "width0_ns", "A1", "width1_ns", "A2", "width2_ns", "A3", "width3_ns"]
    if "two_gaussian" in pulse_family or params_per_gaussian == 3 or (n_channels == 4 and n_params % 12 == 0):
        if not n_gaussians:
            n_gaussians = max(1, n_params // (max(n_channels, 1) * 3))
        names = []
        for channel in range(n_channels):
            for pulse_index in range(n_gaussians):
                names.extend([
                    f"C{channel}_g{pulse_index}_A",
                    f"C{channel}_g{pulse_index}_width_ns",
                    f"C{channel}_g{pulse_index}_center_ns",
                ])
        return names[:n_params]
    if "piecewise" in pulse_family or (n_channels > 0 and n_params > 8 and n_params % n_channels == 0):
        n_time_bins = safe_int(record.get("n_time_bins"), default=0)
        if not n_time_bins:
            l_value = safe_int(record.get("L"), default=0)
            if l_value and n_channels * l_value == n_params:
                n_time_bins = l_value
            else:
                n_time_bins = max(1, n_params // max(n_channels, 1))
        names = []
        for channel in range(n_channels):
            for time_bin in range(n_time_bins):
                names.append(f"C{channel}_bin{time_bin:02d}_A")
        return names[:n_params]
    return [f"param_{idx:02d}" for idx in range(n_params)]


def load_raw_stage1_restart_records(results_root="PiecewiseGRAPEResults_selected_v1", success_threshold=SUCCESS_THRESHOLD):
    results_root = resolve_existing_path(results_root)
    if not results_root.exists():
        raise FileNotFoundError(f"Could not find Stage 2 table or raw Stage 1 results at {results_root}")

    rows = []
    for eta_dir in sorted(results_root.glob("eta_*"), key=lambda p: safe_int(p.name.split("_")[-1])):
        results_path = eta_dir / "results.json"
        if not results_path.exists():
            continue
        eta_index = safe_int(eta_dir.name.split("_")[-1])

        with results_path.open("r") as f:
            records = json.load(f)

        for record in records:
            x = extract_x_opt_scaled_from_record(record)
            names = parameter_names_for_record(record, x)
            final_fidelity = safe_float(record.get("final_fidelity"))
            eta = safe_float(record.get("eta"))
            row = {
                "eta_index": eta_index,
                "eta": eta,
                "eta_over_2pi_GHz": eta / (2 * np.pi * GHz) if math.isfinite(eta) else np.nan,
                "restart": safe_int(record.get("restart")),
                "seed": safe_int(record.get("seed")),
                "seed_type": str(record.get("seed_type", "")),
                "used_physical_seed": safe_bool(record.get("used_physical_seed")),
                "initial_fidelity": safe_float(record.get("initial_fidelity")),
                "final_fidelity": final_fidelity,
                "valid_final_fidelity": math.isfinite(final_fidelity),
                "success_recorded": safe_bool(record.get("success")),
                "success_analysis": math.isfinite(final_fidelity) and final_fidelity >= success_threshold,
                "trap_analysis": not (math.isfinite(final_fidelity) and final_fidelity >= success_threshold),
                "stop_reason": record.get("stop_reason", "missing"),
                "stop_reason_sanitized": str(record.get("stop_reason", "missing")),
                "count": safe_int(record.get("count")),
                "ended_at_bounds": safe_bool(record.get("ended_at_bounds")),
                "source_file": str(results_path),
            }
            for name, value in zip(names, x):
                row[name] = safe_float(value)
            rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    feature_columns = infer_feature_columns_from_dataframe(df)
    set_feature_columns(feature_columns)
    endpoint_valid = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").notna().all(axis=1) if FEATURE_COLUMNS else pd.Series(False, index=df.index)
    df.attrs["input_table_path"] = str(results_root)
    df.attrs["input_source"] = "raw_stage1_results"
    df.attrs["n_input_rows"] = int(len(df))
    df.attrs["n_excluded_missing_endpoints"] = int((~endpoint_valid).sum()) if len(df) else 0
    df.attrs["feature_columns"] = list(FEATURE_COLUMNS)
    return df.loc[endpoint_valid].copy() if len(df) else df


def load_stage2_restart_table(path=RESTART_TABLE_PATH):
    path = resolve_existing_path(path)
    if not path.exists():
        raise FileNotFoundError(f"Stage 2 restart table not found: {path}")

    df_all = pd.read_csv(path)
    feature_columns = infer_feature_columns_from_dataframe(df_all) if FEATURE_COLUMNS is None else list(FEATURE_COLUMNS)
    set_feature_columns(feature_columns)
    if not FEATURE_COLUMNS:
        raise ValueError("Could not infer optimized endpoint columns from Stage 2 restart table.")

    optional_defaults = {
        "eta_index": -1,
        "eta": np.nan,
        "eta_over_2pi_GHz": np.nan,
        "restart": -1,
        "seed": -1,
        "seed_type": "",
        "used_physical_seed": False,
        "initial_fidelity": np.nan,
        "final_fidelity": np.nan,
        "valid_final_fidelity": False,
        "success_recorded": False,
        "success_analysis": False,
        "trap_analysis": False,
        "stop_reason": "missing",
        "stop_reason_sanitized": "missing",
        "count": 0,
        "ended_at_bounds": False,
    }
    for col, default in optional_defaults.items():
        if col not in df_all.columns:
            df_all[col] = default

    for col in FEATURE_COLUMNS + ["eta", "eta_over_2pi_GHz", "initial_fidelity", "final_fidelity", "count"]:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

    df_all["eta_index"] = pd.to_numeric(df_all["eta_index"], errors="coerce").fillna(-1).astype(int)
    df_all["restart"] = pd.to_numeric(df_all["restart"], errors="coerce").fillna(-1).astype(int)
    df_all["seed"] = pd.to_numeric(df_all["seed"], errors="coerce").fillna(-1).astype(int)
    df_all["used_physical_seed"] = df_all["used_physical_seed"].map(safe_bool)
    df_all["success_analysis"] = df_all["success_analysis"].map(safe_bool)
    df_all["success_recorded"] = df_all["success_recorded"].map(safe_bool)
    df_all["trap_analysis"] = df_all["trap_analysis"].map(safe_bool)
    df_all["ended_at_bounds"] = df_all["ended_at_bounds"].map(safe_bool)

    endpoint_valid = df_all[FEATURE_COLUMNS].notna().all(axis=1)
    df_valid = df_all.loc[endpoint_valid].copy()

    df_valid.attrs["input_table_path"] = str(path)
    df_valid.attrs["input_source"] = "stage2_restart_records"
    df_valid.attrs["n_input_rows"] = int(len(df_all))
    df_valid.attrs["n_excluded_missing_endpoints"] = int((~endpoint_valid).sum())
    df_valid.attrs["all_columns"] = list(df_all.columns)
    df_valid.attrs["feature_columns"] = list(FEATURE_COLUMNS)

    print(f"Loaded {len(df_all)} restart rows from {path}")
    print(f"Kept {len(df_valid)} rows with complete optimized endpoint parameters")
    print(f"Endpoint dimension: {len(FEATURE_COLUMNS)}")
    if int((~endpoint_valid).sum()):
        print(f"Excluded {int((~endpoint_valid).sum())} rows with missing endpoint parameters")

    return df_valid


## Feature Normalization and Optional Sign Canonicalization

Main clustering uses raw signed endpoints but normalized features. Amplitudes and widths have different units, so
clustering raw coordinates directly would overweight whichever dimensions have larger numerical ranges.

The sign-canonicalization helper is diagnostic only and is not used for the main results. It can test whether
apparent multiple endpoint families are mainly global sign variants.

In [ ]:
def get_endpoint_feature_matrix(df, normalization="bounds"):
    feature_names = list(FEATURE_COLUMNS or infer_feature_columns_from_dataframe(df))
    if not feature_names:
        raise ValueError("No endpoint feature columns are available for clustering.")
    X_raw = df[feature_names].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

    if normalization == "bounds":
        X_scaled = X_raw.copy()
        for idx, name in enumerate(feature_names):
            kind = parameter_kind(name)
            if kind == "amp":
                lo, hi = AMP_MIN, AMP_MAX
            elif kind == "width":
                lo, hi = WIDTH_MIN_NS, WIDTH_MAX_NS
            elif kind == "center":
                lo, hi = CENTER_MIN_NS, CENTER_MAX_NS
            else:
                finite = X_raw[:, idx][np.isfinite(X_raw[:, idx])]
                lo = float(np.min(finite)) if finite.size else 0.0
                hi = float(np.max(finite)) if finite.size else 1.0
            denom = hi - lo
            X_scaled[:, idx] = (X_raw[:, idx] - lo) / denom if denom > 0 else X_raw[:, idx]

    elif normalization == "zscore":
        require_sklearn()
        X_scaled = StandardScaler().fit_transform(X_raw)

    elif normalization == "none":
        X_scaled = X_raw.copy()

    else:
        raise ValueError("normalization must be 'bounds', 'zscore', or 'none'.")

    return X_raw, X_scaled, feature_names


def canonicalize_endpoint_signs(df):
    # Diagnostic helper: flip all amplitude coordinates if the first large-amplitude coordinate is negative.
    df_canon = df.copy()
    if not AMP_COLUMNS:
        df_canon["sign_canonicalized"] = False
        return df_canon
    amp_values = df_canon[AMP_COLUMNS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

    for row_idx in range(len(df_canon)):
        row = amp_values[row_idx]
        finite = np.where(np.isfinite(row))[0]
        large = [idx for idx in finite if abs(row[idx]) > 1e-9]
        if large and row[large[0]] < 0:
            df_canon.iloc[row_idx, [df_canon.columns.get_loc(col) for col in AMP_COLUMNS]] = -row

    df_canon["sign_canonicalized"] = True
    return df_canon


## Cluster Endpoints

DBSCAN is the default because the number of endpoint families is not known in advance. Agglomerative clustering
is available as a distance-threshold diagnostic. Noise points are labeled `-1`.

In [ ]:
def cluster_one_eta(
    df_eta,
    eta_index,
    method="dbscan",
    normalization="bounds",
    dbscan_eps=0.50,
    dbscan_min_samples=3,
    agglomerative_distance_threshold=0.18,
):
    require_sklearn()
    df_eta = df_eta.copy()
    endpoint_valid = df_eta[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").notna().all(axis=1)
    df_eta = df_eta.loc[endpoint_valid].copy()

    if df_eta.empty:
        summary = {
            "eta_index": int(eta_index),
            "eta": np.nan,
            "eta_over_2pi_GHz": np.nan,
            "method": method,
            "normalization": normalization,
            "n_points": 0,
            "n_clusters": 0,
            "n_noise_points": 0,
            "silhouette_score": np.nan,
            "dbscan_eps": dbscan_eps,
            "dbscan_min_samples": dbscan_min_samples,
            "agglomerative_distance_threshold": agglomerative_distance_threshold,
        }
        df_eta["cluster_label"] = []
        return df_eta, summary

    X_raw, X_scaled, feature_names = get_endpoint_feature_matrix(df_eta, normalization=normalization)

    if method == "dbscan":
        labels = DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples).fit_predict(X_scaled)

    elif method == "agglomerative":
        try:
            model = AgglomerativeClustering(
                n_clusters=None,
                distance_threshold=agglomerative_distance_threshold,
                metric="euclidean",
                linkage="ward",
            )
        except TypeError:
            model = AgglomerativeClustering(
                n_clusters=None,
                distance_threshold=agglomerative_distance_threshold,
                affinity="euclidean",
                linkage="ward",
            )
        labels = model.fit_predict(X_scaled)

    elif method == "none":
        labels = np.zeros(len(df_eta), dtype=int)

    else:
        raise ValueError("method must be 'dbscan', 'agglomerative', or 'none'.")

    labels = np.asarray(labels, dtype=int)
    df_eta["cluster_label"] = labels
    non_noise_labels = sorted(label for label in np.unique(labels) if label != -1)
    n_noise = int(np.sum(labels == -1))

    silhouette = np.nan
    non_noise_mask = labels != -1
    if np.sum(non_noise_mask) >= 3 and len(non_noise_labels) >= 2:
        try:
            silhouette = float(silhouette_score(X_scaled[non_noise_mask], labels[non_noise_mask]))
        except Exception:
            silhouette = np.nan

    eta_value = safe_float(df_eta["eta"].iloc[0]) if "eta" in df_eta else np.nan
    eta_over = safe_float(df_eta["eta_over_2pi_GHz"].iloc[0]) if "eta_over_2pi_GHz" in df_eta else np.nan

    summary = {
        "eta_index": int(eta_index),
        "eta": eta_value,
        "eta_over_2pi_GHz": eta_over,
        "method": method,
        "normalization": normalization,
        "n_points": int(len(df_eta)),
        "n_clusters": int(len(non_noise_labels)),
        "n_noise_points": n_noise,
        "silhouette_score": silhouette,
        "dbscan_eps": float(dbscan_eps),
        "dbscan_min_samples": int(dbscan_min_samples),
        "agglomerative_distance_threshold": float(agglomerative_distance_threshold),
    }

    return df_eta, summary


def cluster_all_eta(
    df,
    method="dbscan",
    normalization="bounds",
    dbscan_eps=0.50,
    dbscan_min_samples=3,
    agglomerative_distance_threshold=0.18,
):
    clustered_parts = []
    summaries = []

    for eta_index in sorted(pd.to_numeric(df["eta_index"], errors="coerce").dropna().astype(int).unique()):
        df_eta = df[df["eta_index"] == eta_index].copy()
        clustered_eta, summary = cluster_one_eta(
            df_eta,
            eta_index=eta_index,
            method=method,
            normalization=normalization,
            dbscan_eps=dbscan_eps,
            dbscan_min_samples=dbscan_min_samples,
            agglomerative_distance_threshold=agglomerative_distance_threshold,
        )
        clustered_parts.append(clustered_eta)
        summaries.append(summary)

    clustered_df = pd.concat(clustered_parts, ignore_index=True) if clustered_parts else pd.DataFrame()
    if not clustered_df.empty:
        clustered_df["is_noise_cluster"] = clustered_df["cluster_label"].astype(int) == -1

        def global_label(row):
            eta_index = int(row["eta_index"])
            label = int(row["cluster_label"])
            if label == -1:
                return f"eta_{eta_index:03d}_noise"
            return f"eta_{eta_index:03d}_cluster_{label:02d}"

        clustered_df["cluster_label_global"] = clustered_df.apply(global_label, axis=1)

    cluster_summary_df = pd.DataFrame(summaries)
    return clustered_df, cluster_summary_df

## Cluster Summaries and Representatives

Each cluster gets two representatives:

- the highest-fidelity endpoint in the cluster,
- the endpoint nearest to the cluster centroid.

These are the candidates to use later as centers for Stage 4 3D Gaussian landscape slices.

In [ ]:
def dominant_value(values, default="missing"):
    values = [str(value) for value in values if str(value) and str(value) != "nan"]
    if not values:
        return default
    return Counter(values).most_common(1)[0][0]


def scaled_distance_to_centroid(group, centroid_raw, normalization="bounds"):
    temp = group.copy()
    X_raw, X_scaled, _ = get_endpoint_feature_matrix(temp, normalization=normalization)
    centroid_df = pd.DataFrame([centroid_raw], columns=FEATURE_COLUMNS)
    _, centroid_scaled, _ = get_endpoint_feature_matrix(centroid_df, normalization=normalization)
    return np.linalg.norm(X_scaled - centroid_scaled[0], axis=1)


def summarize_clusters(clustered_df, success_threshold=SUCCESS_THRESHOLD):
    if clustered_df.empty:
        return pd.DataFrame()

    rows = []
    group_cols = ["eta_index", "cluster_label", "cluster_label_global", "is_noise_cluster"]

    for keys, group in clustered_df.groupby(group_cols, sort=True):
        eta_index, cluster_label, cluster_label_global, is_noise_cluster = keys
        group = group.copy()
        fidelities = pd.to_numeric(group["final_fidelity"], errors="coerce")
        success_mask = fidelities >= success_threshold
        n_points = int(len(group))
        n_success = int(success_mask.fillna(False).sum())

        centroid_values = group[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").mean()
        centroid_raw = [safe_float(centroid_values[col]) for col in FEATURE_COLUMNS]

        valid_fidelity_group = group[fidelities.notna()].copy()
        if not valid_fidelity_group.empty:
            best_idx = valid_fidelity_group["final_fidelity"].astype(float).idxmax()
            best_row = valid_fidelity_group.loc[best_idx]
        else:
            best_row = group.iloc[0]

        distances = scaled_distance_to_centroid(group, centroid_raw, normalization="bounds")
        centroid_row = group.iloc[int(np.argmin(distances))]

        row = {
            "eta_index": int(eta_index),
            "eta": safe_float(group["eta"].iloc[0]),
            "eta_over_2pi_GHz": safe_float(group["eta_over_2pi_GHz"].iloc[0]),
            "cluster_label": int(cluster_label),
            "cluster_label_global": str(cluster_label_global),
            "is_noise_cluster": bool(is_noise_cluster),
            "n_points": n_points,
            "n_success": n_success,
            "success_rate": n_success / n_points if n_points else np.nan,
            "best_fidelity": safe_float(fidelities.max()),
            "mean_fidelity": safe_float(fidelities.mean()),
            "median_fidelity": safe_float(fidelities.median()),
            "min_fidelity": safe_float(fidelities.min()),
            "max_fidelity": safe_float(fidelities.max()),
            "physical_seed_count": int(group["used_physical_seed"].map(safe_bool).sum()),
            "random_seed_count": int((~group["used_physical_seed"].map(safe_bool)).sum()),
            "physical_seed_fraction": float(group["used_physical_seed"].map(safe_bool).mean()) if n_points else np.nan,
            "mean_iterations": safe_float(pd.to_numeric(group["count"], errors="coerce").mean()),
            "median_iterations": safe_float(pd.to_numeric(group["count"], errors="coerce").median()),
            "ended_at_bounds_count": int(group["ended_at_bounds"].map(safe_bool).sum()),
            "dominant_stop_reason": dominant_value(group.get("stop_reason_sanitized", group.get("stop_reason", []))),
        }

        for name, value in zip(FEATURE_COLUMNS, centroid_raw):
            row[f"centroid_{name}"] = value

        row.update({
            "rep_best_restart": safe_int(best_row.get("restart"), default=-1),
            "rep_best_seed": safe_int(best_row.get("seed"), default=-1),
            "rep_best_final_fidelity": safe_float(best_row.get("final_fidelity")),
            "rep_best_used_physical_seed": safe_bool(best_row.get("used_physical_seed")),
        })
        for name in FEATURE_COLUMNS:
            row[f"rep_best_{name}"] = safe_float(best_row.get(name))

        row.update({
            "rep_centroid_restart": safe_int(centroid_row.get("restart"), default=-1),
            "rep_centroid_seed": safe_int(centroid_row.get("seed"), default=-1),
            "rep_centroid_final_fidelity": safe_float(centroid_row.get("final_fidelity")),
        })
        for name in FEATURE_COLUMNS:
            row[f"rep_centroid_{name}"] = safe_float(centroid_row.get(name))

        rows.append(row)

    return pd.DataFrame(rows)


def summarize_cluster_counts_by_eta(cluster_summary_df, success_threshold=SUCCESS_THRESHOLD):
    if cluster_summary_df.empty:
        return pd.DataFrame()

    rows = []
    for eta_index, group in cluster_summary_df.groupby("eta_index", sort=True):
        ordinary = group[~group["is_noise_cluster"].map(safe_bool)].copy()
        noise = group[group["is_noise_cluster"].map(safe_bool)].copy()
        high = ordinary[ordinary["best_fidelity"] >= success_threshold]
        low = ordinary[ordinary["best_fidelity"] < success_threshold]
        mixed = ordinary[(ordinary["n_success"] > 0) & (ordinary["n_success"] < ordinary["n_points"])]

        largest_row = None
        if len(ordinary):
            largest_row = ordinary.loc[ordinary["n_points"].idxmax()]

        row = {
            "eta_index": int(eta_index),
            "eta": safe_float(group["eta"].iloc[0]),
            "eta_over_2pi_GHz": safe_float(group["eta_over_2pi_GHz"].iloc[0]),
            "n_clusters_total": int(len(ordinary)),
            "n_noise_points": int(noise["n_points"].sum()) if len(noise) else 0,
            "n_high_fidelity_clusters": int(len(high)),
            "n_low_fidelity_clusters": int(len(low)),
            "n_mixed_clusters": int(len(mixed)),
            "largest_cluster_size": int(largest_row["n_points"]) if largest_row is not None else 0,
            "largest_cluster_success_rate": safe_float(largest_row["success_rate"]) if largest_row is not None else np.nan,
            "best_cluster_fidelity": safe_float(ordinary["best_fidelity"].max()) if len(ordinary) else np.nan,
            "physical_seed_clusters": int((ordinary["physical_seed_count"] > 0).sum()) if len(ordinary) else 0,
            "random_seed_clusters": int((ordinary["random_seed_count"] > 0).sum()) if len(ordinary) else 0,
        }
        rows.append(row)

    return pd.DataFrame(rows)


def select_representative_optima(cluster_summary_df):
    if cluster_summary_df.empty:
        return pd.DataFrame()
    # Keep noise representatives too. Noise points are not counted as ordinary clusters,
    # but a high-fidelity endpoint labeled as DBSCAN noise may still be useful for Stage 4 slicing.
    representative_df = cluster_summary_df.copy()
    representative_df = representative_df.sort_values(
        ["eta_index", "is_noise_cluster", "rep_best_final_fidelity", "n_points"],
        ascending=[True, True, False, False],
    ).reset_index(drop=True)
    representative_df["representative_rank_within_eta"] = representative_df.groupby("eta_index").cumcount() + 1
    ordinary_mask = ~representative_df["is_noise_cluster"].map(safe_bool)
    representative_df["ordinary_representative_rank_within_eta"] = np.nan
    representative_df.loc[ordinary_mask, "ordinary_representative_rank_within_eta"] = (
        representative_df.loc[ordinary_mask].groupby("eta_index").cumcount() + 1
    )
    return representative_df

## Sensitivity Sweep

DBSCAN cluster counts depend on `eps` and `min_samples`. The sensitivity table records how cluster counts change
across reasonable settings so the final interpretation is not tied to a single arbitrary threshold.

In [ ]:
def clustering_sensitivity_sweep(
    df,
    eps_values=(0.08, 0.10, 0.12, 0.15, 0.18, 0.22, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70),
    min_samples_values=(2, 3, 5),
    normalization="bounds",
):
    rows = []
    for eps in eps_values:
        for min_samples in min_samples_values:
            clustered_df, _ = cluster_all_eta(
                df,
                method="dbscan",
                normalization=normalization,
                dbscan_eps=eps,
                dbscan_min_samples=min_samples,
            )
            cluster_summary_df = summarize_clusters(clustered_df, success_threshold=SUCCESS_THRESHOLD)
            cluster_count_df = summarize_cluster_counts_by_eta(cluster_summary_df, success_threshold=SUCCESS_THRESHOLD)

            for _, row in cluster_count_df.iterrows():
                rows.append({
                    "eps": float(eps),
                    "min_samples": int(min_samples),
                    "eta_index": safe_int(row.get("eta_index")),
                    "n_clusters_total": safe_int(row.get("n_clusters_total")),
                    "n_noise_points": safe_int(row.get("n_noise_points")),
                    "n_high_fidelity_clusters": safe_int(row.get("n_high_fidelity_clusters")),
                    "n_low_fidelity_clusters": safe_int(row.get("n_low_fidelity_clusters")),
                    "largest_cluster_size": safe_int(row.get("largest_cluster_size")),
                })

    return pd.DataFrame(rows)

## Plotting Functions

In [ ]:
def plot_cluster_count_vs_eta(cluster_count_df, figures_dir):
    if cluster_count_df.empty:
        record_skipped_figure("cluster_count_vs_eta.png", "cluster_count_df is empty")
        record_skipped_figure("high_low_cluster_count_vs_eta.png", "cluster_count_df is empty")
        return

    x = cluster_count_df["eta_over_2pi_GHz"]

    fig, ax = plt.subplots(figsize=(figure_width, figure_height))
    ax.plot(x, cluster_count_df["n_clusters_total"], marker="o", label="total clusters")
    ax.plot(x, cluster_count_df["n_high_fidelity_clusters"], marker="o", label="high-fidelity clusters")
    ax.plot(x, cluster_count_df["n_low_fidelity_clusters"], marker="o", label="low-fidelity clusters")
    ax.set_xlabel("Crosstalk eta / (2 pi GHz)")
    ax.set_ylabel("Cluster count")
    ax.set_title("Endpoint cluster count versus crosstalk")
    ax.grid(True, alpha=0.3)
    ax.legend()
    save_figure(fig, Path(figures_dir) / "cluster_count_vs_eta.png")

    fig, ax = plt.subplots(figsize=(figure_width, figure_height))
    ax.plot(x, cluster_count_df["n_high_fidelity_clusters"], marker="o", label="high fidelity")
    ax.plot(x, cluster_count_df["n_low_fidelity_clusters"], marker="o", label="low fidelity")
    ax.set_xlabel("Crosstalk eta / (2 pi GHz)")
    ax.set_ylabel("Cluster count")
    ax.set_title("High- and low-fidelity endpoint clusters")
    ax.grid(True, alpha=0.3)
    ax.legend()
    save_figure(fig, Path(figures_dir) / "high_low_cluster_count_vs_eta.png")


def plot_pca_clusters_for_eta(clustered_df, eta_index, figures_dir):
    require_sklearn()
    df_eta = clustered_df[clustered_df["eta_index"] == int(eta_index)].copy()
    suffix = eta_figure_suffix(clustered_df, eta_index)
    filename = f"pca_clusters_{suffix}.png"
    if len(df_eta) < 3:
        record_skipped_figure(filename, "fewer than three endpoints")
        return

    _, X_scaled, _ = get_endpoint_feature_matrix(df_eta, normalization="bounds")
    if X_scaled.shape[1] < 2:
        record_skipped_figure(filename, "fewer than two endpoint dimensions")
        return

    coords = PCA(n_components=2).fit_transform(X_scaled)
    labels = df_eta["cluster_label"].astype(int).to_numpy()
    success = df_eta["success_analysis"].map(safe_bool).to_numpy()
    physical = df_eta["used_physical_seed"].map(safe_bool).to_numpy()

    fig, ax = plt.subplots(figsize=(figure_width, figure_height))
    unique_labels = sorted(np.unique(labels))
    cmap = plt.get_cmap("tab20", max(len(unique_labels), 1))

    for color_idx, label in enumerate(unique_labels):
        label_mask = labels == label
        cluster_name = "noise" if label == -1 else f"cluster {label}"
        for is_success, marker in [(True, "o"), (False, "x")]:
            mask = label_mask & (success == is_success)
            if not np.any(mask):
                continue
            ax.scatter(
                coords[mask, 0],
                coords[mask, 1],
                color=cmap(color_idx),
                marker=marker,
                alpha=0.8,
                label=f"{cluster_name}, {'success' if is_success else 'not success'}",
            )

    if np.any(physical):
        ax.scatter(
            coords[physical, 0],
            coords[physical, 1],
            facecolors="none",
            edgecolors="black",
            linewidths=1.2,
            s=95,
            label="physical seed",
        )

    eta_over = safe_float(df_eta["eta_over_2pi_GHz"].iloc[0])
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")
    ax.set_title(f"Endpoint PCA clusters, eta={eta_over:.3f}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    save_figure(fig, Path(figures_dir) / filename)


def plot_pca_clusters_all_eta(clustered_df, figures_dir):
    require_sklearn()
    filename = "pca_clusters_all_eta.png"
    if len(clustered_df) < 3:
        record_skipped_figure(filename, "fewer than three endpoints")
        return

    _, X_scaled, _ = get_endpoint_feature_matrix(clustered_df, normalization="bounds")
    coords = PCA(n_components=2).fit_transform(X_scaled)
    eta_indices = clustered_df["eta_index"].astype(int).to_numpy()
    success = clustered_df["success_analysis"].map(safe_bool).to_numpy()

    fig, ax = plt.subplots(figsize=(figure_width, figure_height))
    unique_eta = sorted(np.unique(eta_indices))
    cmap = plt.get_cmap("viridis", max(len(unique_eta), 1))
    for color_idx, eta_index in enumerate(unique_eta):
        eta_mask = eta_indices == eta_index
        for is_success, marker in [(True, "o"), (False, "x")]:
            mask = eta_mask & (success == is_success)
            if not np.any(mask):
                continue
            ax.scatter(
                coords[mask, 0],
                coords[mask, 1],
                color=cmap(color_idx),
                marker=marker,
                alpha=0.65,
                label=f"eta {eta_index:03d}, {'success' if is_success else 'not success'}",
            )
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")
    ax.set_title("All eta endpoint PCA projection")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, ncol=3)
    save_figure(fig, Path(figures_dir) / filename)


def plot_success_rate_by_cluster(cluster_summary_df, eta_index, figures_dir):
    suffix = eta_figure_suffix(cluster_summary_df, eta_index) if "cluster_label" in cluster_summary_df else f"eta_{int(eta_index):03d}"
    filename = f"success_rate_by_cluster_{suffix}.png"
    df_eta = cluster_summary_df[cluster_summary_df["eta_index"] == int(eta_index)].copy()
    if df_eta.empty:
        record_skipped_figure(filename, "no cluster summary rows for eta")
        return

    df_eta = df_eta.sort_values(["is_noise_cluster", "cluster_label"])
    labels = [
        "noise" if safe_bool(row["is_noise_cluster"]) else f"c{int(row['cluster_label'])}"
        for _, row in df_eta.iterrows()
    ]
    colors = ["lightgray" if safe_bool(v) else "steelblue" for v in df_eta["is_noise_cluster"]]

    fig, ax = plt.subplots(figsize=(figure_width, figure_height))
    ax.bar(labels, df_eta["success_rate"], color=colors)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Success rate")
    eta_over = safe_float(df_eta["eta_over_2pi_GHz"].iloc[0])
    ax.set_title(f"Success rate by endpoint cluster, eta={eta_over:.3f}")
    ax.grid(True, axis="y", alpha=0.3)
    for idx, (_, row) in enumerate(df_eta.iterrows()):
        ax.text(idx, safe_float(row["success_rate"], 0) + 0.03, f"n={int(row['n_points'])}", ha="center", fontsize=9)
    save_figure(fig, Path(figures_dir) / filename)


def plot_representative_parameters_vs_eta(representative_df, figures_dir):
    filename = "representative_parameters_vs_eta.png"
    if representative_df.empty:
        record_skipped_figure(filename, "representative_df is empty")
        return

    best_per_eta = representative_df.sort_values(
        ["eta_index", "is_noise_cluster", "rep_best_final_fidelity"],
        ascending=[True, True, False],
    ).groupby("eta_index", as_index=False).head(1)
    best_per_eta = best_per_eta[~best_per_eta["is_noise_cluster"].map(safe_bool)].copy()
    if best_per_eta.empty:
        record_skipped_figure(filename, "no non-noise representatives")
        return

    x = best_per_eta["eta_over_2pi_GHz"]
    groups = [
        ("amplitudes", [col for col in FEATURE_COLUMNS if parameter_kind(col) == "amp"], "Amplitude"),
        ("widths", [col for col in FEATURE_COLUMNS if parameter_kind(col) == "width"], "Width (ns)"),
        ("centers", [col for col in FEATURE_COLUMNS if parameter_kind(col) == "center"], "Center (ns)"),
    ]
    groups = [(name, cols, ylabel) for name, cols, ylabel in groups if cols]
    if not groups:
        record_skipped_figure(filename, "no plottable representative parameter groups")
        return

    fig, axes = plt.subplots(len(groups), 1, figsize=(figure_width, 3.5 * len(groups)), sharex=True)
    if len(groups) == 1:
        axes = [axes]

    for ax, (group_name, columns, ylabel) in zip(axes, groups):
        for column in columns:
            rep_column = f"rep_best_{column}"
            if rep_column in best_per_eta.columns:
                ax.plot(x, best_per_eta[rep_column], marker="o", label=column)
        ax.set_ylabel(ylabel)
        ax.set_title(f"Best representative endpoint {group_name} versus eta")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, ncol=2)

    axes[-1].set_xlabel("Crosstalk eta / (2 pi GHz)")
    plt.tight_layout()
    save_figure(fig, Path(figures_dir) / filename)


def plot_physical_vs_random_cluster_counts(cluster_count_df, figures_dir):
    filename = "physical_vs_random_cluster_counts.png"
    if cluster_count_df.empty:
        record_skipped_figure(filename, "cluster_count_df is empty")
        return

    x = cluster_count_df["eta_over_2pi_GHz"]
    fig, ax = plt.subplots(figsize=(figure_width, figure_height))
    ax.plot(x, cluster_count_df["physical_seed_clusters"], marker="o", label="clusters containing physical seeds")
    ax.plot(x, cluster_count_df["random_seed_clusters"], marker="o", label="clusters containing random seeds")
    ax.set_xlabel("Crosstalk eta / (2 pi GHz)")
    ax.set_ylabel("Cluster count")
    ax.set_title("Physical versus random seed cluster occupancy")
    ax.grid(True, alpha=0.3)
    ax.legend()
    save_figure(fig, Path(figures_dir) / filename)

## Save Tables and Report

In [ ]:
def save_stage3_outputs(
    clustered_df,
    cluster_summary_df,
    cluster_count_df,
    representative_df,
    sensitivity_df,
    validation_report,
    out_dir=STAGE3_OUT,
):
    tables_dir, figures_dir = ensure_output_dirs(out_dir)

    clustered_df.to_csv(tables_dir / "clustered_restart_records.csv", index=False)
    cluster_summary_df.to_csv(tables_dir / "cluster_summary_by_eta.csv", index=False)
    cluster_summary_df.to_json(tables_dir / "cluster_summary_by_eta.json", orient="records", indent=2)
    cluster_count_df.to_csv(tables_dir / "cluster_count_by_eta.csv", index=False)
    cluster_count_df.to_json(tables_dir / "cluster_count_by_eta.json", orient="records", indent=2)
    representative_df.to_csv(tables_dir / "representative_optima.csv", index=False)
    representative_df.to_json(tables_dir / "representative_optima.json", orient="records", indent=2)
    sensitivity_df.to_csv(tables_dir / "clustering_sensitivity.csv", index=False)

    parquet_saved = False
    parquet_error = None
    try:
        clustered_df.to_parquet(tables_dir / "clustered_restart_records.parquet", index=False)
        parquet_saved = True
    except Exception as exc:
        parquet_error = str(exc)

    validation_report["parquet_saved"] = parquet_saved
    if parquet_error:
        validation_report["parquet_error"] = parquet_error

    with (tables_dir / "validation_report_stage3.json").open("w") as f:
        json.dump(make_json_safe(validation_report), f, indent=2)

    return {
        "tables_dir": tables_dir,
        "figures_dir": figures_dir,
        "parquet_saved": parquet_saved,
        "parquet_error": parquet_error,
    }


def assess_cluster_count_trend(cluster_count_df):
    if cluster_count_df.empty or len(cluster_count_df) < 2:
        return "insufficient data"
    first = safe_float(cluster_count_df.sort_values("eta_index")["n_clusters_total"].iloc[0])
    last = safe_float(cluster_count_df.sort_values("eta_index")["n_clusters_total"].iloc[-1])
    if last > first:
        return "increases"
    if last < first:
        return "decreases"
    return "roughly unchanged"


def assess_seed_overlap(cluster_summary_df):
    ordinary = cluster_summary_df[~cluster_summary_df["is_noise_cluster"].map(safe_bool)].copy()
    noise = cluster_summary_df[cluster_summary_df["is_noise_cluster"].map(safe_bool)].copy()
    if ordinary.empty and noise.empty:
        return "insufficient data"
    both = int(((ordinary["physical_seed_count"] > 0) & (ordinary["random_seed_count"] > 0)).sum()) if len(ordinary) else 0
    physical_only = int(((ordinary["physical_seed_count"] > 0) & (ordinary["random_seed_count"] == 0)).sum()) if len(ordinary) else 0
    random_only = int(((ordinary["physical_seed_count"] == 0) & (ordinary["random_seed_count"] > 0)).sum()) if len(ordinary) else 0
    noise_physical = int(noise["physical_seed_count"].sum()) if len(noise) else 0
    noise_random = int(noise["random_seed_count"].sum()) if len(noise) else 0
    return (
        f"Ordinary clusters: {both} contain both physical and random seeds; "
        f"{physical_only} physical-only; {random_only} random-only. "
        f"DBSCAN noise endpoints: {noise_physical} physical-seed endpoints and {noise_random} random-seed endpoints."
    )


def write_stage3_report(
    out_dir,
    input_table_path,
    validation_report,
    cluster_count_df,
    cluster_summary_df,
    method,
    normalization,
    dbscan_eps,
    dbscan_min_samples,
    agglomerative_distance_threshold,
):
    out_dir = Path(out_dir)
    eta0_high = np.nan
    high_eta_high = np.nan
    if not cluster_count_df.empty:
        ordered = cluster_count_df.sort_values("eta_index")
        eta0_high = safe_int(ordered["n_high_fidelity_clusters"].iloc[0], default=0)
        high_eta_high = safe_int(ordered["n_high_fidelity_clusters"].iloc[-1], default=0)

    trend = assess_cluster_count_trend(cluster_count_df)
    seed_overlap = assess_seed_overlap(cluster_summary_df)

    lines = [
        "# Gaussian GRAPE Stage 3 Endpoint Clustering Report",
        "",
        f"Input table: `{input_table_path}`",
        f"Endpoints loaded for clustering: {validation_report.get('n_endpoint_rows_loaded')}",
        f"Endpoints excluded due to missing parameters: {validation_report.get('n_excluded_missing_endpoints')}",
        f"Clustering method: `{method}`",
        f"Normalization: `{normalization}`",
        f"DBSCAN eps: `{dbscan_eps}`",
        f"DBSCAN min_samples: `{dbscan_min_samples}`",
        f"Agglomerative distance threshold: `{agglomerative_distance_threshold}`",
        "",
        "## Cluster Count Trend",
        "",
        f"Cluster count trend versus eta: **{trend}**.",
        f"High-fidelity clusters at eta=0: **{eta0_high}**.",
        f"High-fidelity clusters at highest eta: **{high_eta_high}**.",
        "",
        "## Physical Versus Random Seeds",
        "",
        seed_overlap,
        "",
        "## Notes",
        "",
        "- Clustering estimates endpoint families, not mathematically exact optima.",
        "- DBSCAN results depend on eps/min_samples, so inspect `tables/clustering_sensitivity.csv`.",
        "- Sign-related clusters may be physically equivalent pulse solutions.",
        "- `representative_optima` includes ordinary cluster representatives and DBSCAN-noise representatives; noise representatives are useful diagnostics when high-fidelity endpoints are isolated.",
        "- Representative optima from this stage should be used as centers for later Stage 4 3D endpoint landscape slices.",
        "- Hessian analysis and 3D landscape slices come later.",
    ]

    report_path = out_dir / "stage3_report.md"
    with report_path.open("w") as f:
        f.write("\n".join(lines) + "\n")
    return report_path

## Main Stage 3 Pipeline

In [ ]:
def run_stage3_clustering_analysis(
    stage2_dir=STAGE2_ANALYSIS_DIR,
    out_dir=STAGE3_OUT,
    success_threshold=SUCCESS_THRESHOLD,
    method="dbscan",
    normalization="bounds",
    dbscan_eps=0.50,
    dbscan_min_samples=3,
    agglomerative_distance_threshold=0.18,
    run_sensitivity=True,
):
    require_sklearn()
    FIGURE_STATUS.clear()

    stage2_dir = resolve_stage2_dir(stage2_dir)
    out_dir = resolve_stage3_out(out_dir, stage2_dir=stage2_dir)
    tables_dir, figures_dir = ensure_output_dirs(out_dir)
    restart_path = stage2_dir / "tables" / "restart_records.csv"

    input_source = "stage2_restart_records"
    if restart_path.exists():
        df_input = load_stage2_restart_table(restart_path)
    else:
        print(f"Stage 2 restart table not found at {restart_path}. Falling back to raw Stage 1 results.")
        df_input = load_raw_stage1_restart_records(success_threshold=success_threshold)
        input_source = df_input.attrs.get("input_source", "raw_stage1_results")

    n_input_rows = int(df_input.attrs.get("n_input_rows", len(df_input)))
    n_excluded = int(df_input.attrs.get("n_excluded_missing_endpoints", 0))
    input_table_path = df_input.attrs.get("input_table_path", str(restart_path))

    clustered_df, cluster_method_summary_df = cluster_all_eta(
        df_input,
        method=method,
        normalization=normalization,
        dbscan_eps=dbscan_eps,
        dbscan_min_samples=dbscan_min_samples,
        agglomerative_distance_threshold=agglomerative_distance_threshold,
    )
    cluster_summary_df = summarize_clusters(clustered_df, success_threshold=success_threshold)
    cluster_count_df = summarize_cluster_counts_by_eta(cluster_summary_df, success_threshold=success_threshold)
    representative_df = select_representative_optima(cluster_summary_df)

    if run_sensitivity:
        sensitivity_df = clustering_sensitivity_sweep(df_input, normalization=normalization)
    else:
        sensitivity_df = pd.DataFrame()

    eta_indices = sorted(clustered_df["eta_index"].astype(int).unique()) if not clustered_df.empty else []
    warnings = []
    notes = []
    if N_ETA_EXPECTED is not None and len(eta_indices) != int(N_ETA_EXPECTED):
        warnings.append(f"Expected {N_ETA_EXPECTED} eta values but found {len(eta_indices)}.")
    if clustered_df.empty:
        warnings.append("No clustered endpoint rows were produced.")
    if not warnings:
        notes.append("Stage 3 clustering input validation passed.")

    plot_cluster_count_vs_eta(cluster_count_df, figures_dir)
    if eta_indices:
        selected_eta = sorted(set([eta_indices[0], eta_indices[len(eta_indices) // 2], eta_indices[-1]]))
        for eta_index in selected_eta:
            plot_success_rate_by_cluster(cluster_summary_df, eta_index, figures_dir)
            plot_pca_clusters_for_eta(clustered_df, eta_index, figures_dir)
    else:
        for name in [
            "success_rate_by_cluster_eta_000.png",
            "success_rate_by_cluster_mid_eta.png",
            "success_rate_by_cluster_eta_008.png",
            "pca_clusters_eta_000.png",
            "pca_clusters_mid_eta.png",
            "pca_clusters_eta_008.png",
        ]:
            record_skipped_figure(name, "no eta indices found")

    plot_pca_clusters_all_eta(clustered_df, figures_dir)
    plot_representative_parameters_vs_eta(representative_df, figures_dir)
    plot_physical_vs_random_cluster_counts(cluster_count_df, figures_dir)

    validation_report = {
        "stage": "stage3_endpoint_clustering",
        "input_source": input_source,
        "input_table_path": str(input_table_path),
        "output_dir": str(out_dir),
        "success_threshold": float(success_threshold),
        "method": method,
        "normalization": normalization,
        "dbscan_eps": float(dbscan_eps),
        "dbscan_min_samples": int(dbscan_min_samples),
        "agglomerative_distance_threshold": float(agglomerative_distance_threshold),
        "n_input_rows": n_input_rows,
        "n_endpoint_rows_loaded": int(len(df_input)),
        "n_excluded_missing_endpoints": n_excluded,
        "n_eta_expected": None if N_ETA_EXPECTED is None else int(N_ETA_EXPECTED),
        "n_eta_found": int(len(eta_indices)),
        "eta_indices_found": [int(value) for value in eta_indices],
        "n_clustered_rows": int(len(clustered_df)),
        "n_cluster_summary_rows": int(len(cluster_summary_df)),
        "n_representative_rows": int(len(representative_df)),
        "run_sensitivity": bool(run_sensitivity),
        "n_sensitivity_rows": int(len(sensitivity_df)),
        "figures": list(FIGURE_STATUS),
        "warnings": warnings,
        "notes": notes,
        "cluster_method_summary": cluster_method_summary_df.to_dict(orient="records"),
    }

    save_info = save_stage3_outputs(
        clustered_df=clustered_df,
        cluster_summary_df=cluster_summary_df,
        cluster_count_df=cluster_count_df,
        representative_df=representative_df,
        sensitivity_df=sensitivity_df,
        validation_report=validation_report,
        out_dir=out_dir,
    )

    report_path = write_stage3_report(
        out_dir=out_dir,
        input_table_path=input_table_path,
        validation_report=validation_report,
        cluster_count_df=cluster_count_df,
        cluster_summary_df=cluster_summary_df,
        method=method,
        normalization=normalization,
        dbscan_eps=dbscan_eps,
        dbscan_min_samples=dbscan_min_samples,
        agglomerative_distance_threshold=agglomerative_distance_threshold,
    )

    print(f"Stage 3 clustering complete. Outputs saved to {out_dir}")
    print(f"Report: {report_path}")

    return {
        "df_input": df_input,
        "clustered_df": clustered_df,
        "cluster_summary_df": cluster_summary_df,
        "cluster_count_df": cluster_count_df,
        "representative_df": representative_df,
        "sensitivity_df": sensitivity_df,
        "validation_report": validation_report,
        "save_info": save_info,
        "report_path": report_path,
    }

## Run Stage 3

This cell runs post-processing only. It does not run any new quantum-control optimization.

In [ ]:
stage3 = run_stage3_clustering_analysis(
    stage2_dir=STAGE2_ANALYSIS_DIR,
    out_dir=STAGE3_OUT,
    success_threshold=SUCCESS_THRESHOLD,
    method="dbscan",
    normalization="bounds",
    dbscan_eps=0.50,
    dbscan_min_samples=3,
    run_sensitivity=True,
)

clustered_df = stage3["clustered_df"]
cluster_summary_df = stage3["cluster_summary_df"]
cluster_count_df = stage3["cluster_count_df"]
representative_df = stage3["representative_df"]

display(cluster_count_df)
display(cluster_summary_df.head())
display(representative_df.head())